# Local Icechunk reference generation — IPSL CMIP7 test files

Received an early look at IPSL CMIP7 files located here: https://thredds-su.ipsl.fr/thredds/catalog/ESGF-Test/incoming/IPSL/piControl_test2/catalog.html

**Goal**: Double check that we can create an icechunk store from local files here, basically QC for the ref generation.

Four already-downloaded files (~1.6 GB) live **outside the repository** and are
untracked (`*.nc` is gitignored). They are referenced by absolute path:

```
/Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data/
```

Three shapes are covered, because each stresses something different:

| shape | files | what it tests |
|---|---|---|
| multi-file concat | 2 × `ficeberg` (3hr) | concatenation across a file boundary |
| single file | 1 × `thetao` (mon, 1.46 GB) | a second dimension coordinate (`olevel`) |
| `fx` — no time | 1 × `areacello` | a dataset with nothing to concatenate |

In [3]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

import icechunk as ic
import numpy as np
import virtualizarr
import xarray as xr

import cmip7_virtualization as cv
from cmip7_virtualization.storage import (
    authorize_prefixes_from_registry,
    local_url_prefix,
    vccs_from_registry,
)
from cmip7_virtualization.virtualize import (
    DEFAULT_LOADABLE_VARIABLES,
    as_url,
    virtualize_from_urls,
)

print("icechunk    ", ic.__version__)
print("virtualizarr", virtualizarr.__version__)
print("xarray      ", xr.__version__)

icechunk     2.1.1
virtualizarr 2.1.3.dev6+gcb2912e65
xarray       2025.10.1


In [4]:
# The NetCDF files live in the MAIN checkout, not in this worktree, and are never copied.
DATA = Path(
    "/Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data"
)

# Icechunk stores go to a temp dir, never into notebooks/. Override with an env var to
# keep them around (e.g. CMIP7_LOCAL_STORE_ROOT=refs/local-ipsl).
STORE_ROOT = Path(
    os.environ.get(
        "CMIP7_LOCAL_STORE_ROOT", Path(tempfile.gettempdir()) / "cmip7-ipsl-icechunk"
    )
)
# Cleared on every run: `to_icechunk` writes the root group and raises
# ContainsGroupError against a store that already holds one, so re-running the notebook
# over a previous store fails rather than overwriting.
shutil.rmtree(STORE_ROOT, ignore_errors=True)
STORE_ROOT.mkdir(parents=True, exist_ok=True)

GROUPS = {
    "ficeberg": sorted(DATA.glob("ficeberg_*.nc")),  # multi-file concat, 3hr
    "thetao": sorted(DATA.glob("thetao_*.nc")),  # single file, mon, 4-D
    "areacello": sorted(DATA.glob("areacello_*.nc")),  # fx, no time dimension
}

for name, paths in GROUPS.items():
    assert paths, f"no files found for {name} in {DATA}"
    for p in paths:
        print(f"{p.stat().st_size / 1e6:9.1f} MB  {p.name}")
print(f"\nstore root: {STORE_ROOT}")

     82.9 MB  ficeberg_tavg-ol-hxy-sea_3hr_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185001010130-185004302230.nc
     84.8 MB  ficeberg_tavg-ol-hxy-sea_3hr_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185005010130-185008312230.nc
   1458.0 MB  thetao_tavg-ol-hxy-sea_mon_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185001-185912.nc
      4.9 MB  areacello_ti-u-hxy-u_fx_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1.nc

store root: /var/folders/3b/rlk8rxgs1xx44kmtzfrwlvth0000gn/T/cmip7-ipsl-icechunk


## CMIP7 branded-variable filenames

These filenames are **not** CMIP6-shaped. CMIP6 was

```
<variable_id>_<table_id>_<source_id>_<experiment_id>_<variant_label>_<grid_label>[_<time_range>].nc
```

CMIP7 (`drs_specs = MIP-DRS7`) drops `table_id` entirely and inserts a *branding
suffix*, a bare `frequency`, and a `region`:

```
<variable_id>_<branding_suffix>_<frequency>_<region>_<grid_label>_<source_id>_<experiment_id>_<variant_label>[_<time_range>].nc
```

The branding suffix is itself four dash-separated labels —
`<temporal>-<vertical>-<horizontal>-<area>`. So `tavg-ol-hxy-sea` is a time **av**erage
on **o**cean **l**evels over a **h**orizontal **x**–**y** grid, masked to **sea**; and
`ti-u-hxy-u` is **t**ime-**i**ndependent with vertical and area **u**nspecified.

In [6]:
DRS7_FIELDS = [
    "variable_id",
    "branding_suffix",
    "frequency",
    "region",
    "grid_label",
    "source_id",
    "experiment_id",
    "variant_label",
    "time_range",  # absent for fx
]


def parse_drs7(path):
    parts = Path(path).stem.split("_")
    return dict(zip(DRS7_FIELDS, parts))


for paths in GROUPS.values():
    for p in paths:
        print(parse_drs7(p))

{'variable_id': 'ficeberg', 'branding_suffix': 'tavg-ol-hxy-sea', 'frequency': '3hr', 'region': 'glb', 'grid_label': 'g112', 'source_id': 'IPSLCM6-ESMCO2', 'experiment_id': 'piControl', 'variant_label': 'r1i1p1f1', 'time_range': '185001010130-185004302230'}
{'variable_id': 'ficeberg', 'branding_suffix': 'tavg-ol-hxy-sea', 'frequency': '3hr', 'region': 'glb', 'grid_label': 'g112', 'source_id': 'IPSLCM6-ESMCO2', 'experiment_id': 'piControl', 'variant_label': 'r1i1p1f1', 'time_range': '185005010130-185008312230'}
{'variable_id': 'thetao', 'branding_suffix': 'tavg-ol-hxy-sea', 'frequency': 'mon', 'region': 'glb', 'grid_label': 'g112', 'source_id': 'IPSLCM6-ESMCO2', 'experiment_id': 'piControl', 'variant_label': 'r1i1p1f1', 'time_range': '185001-185912'}
{'variable_id': 'areacello', 'branding_suffix': 'ti-u-hxy-u', 'frequency': 'fx', 'region': 'glb', 'grid_label': 'g112', 'source_id': 'IPSLCM6-ESMCO2', 'experiment_id': 'piControl', 'variant_label': 'r1i1p1f1'}


Filename parsing is only ever a convenience, though. The authoritative identifiers are
in the global attributes, and CMIP7 files carry the branding there explicitly.

In [7]:
ds_attrs = xr.open_dataset(GROUPS["thetao"][0], decode_times=False, engine="h5netcdf")
keys = [
    "mip_era",
    "drs_specs",
    "activity_id",
    "institution_id",
    "source_id",
    "experiment_id",
    "variant_label",
    "grid_label",
    "nominal_resolution",
    "region",
    "frequency",
    "realm",
    "variable_id",
    "branded_variable",
    "branding_suffix",
    "temporal_label",
    "vertical_label",
    "horizontal_label",
    "area_label",
    "data_specs_version",
]
for k in keys:
    print(f"{k:22s} {ds_attrs.attrs.get(k, '<absent>')}")
print(
    f"{'table_id':22s} {ds_attrs.attrs.get('table_id', '<absent>')}   <-- gone in CMIP7"
)
ds_attrs.close()

mip_era                CMIP7
drs_specs              MIP-DRS7
activity_id            CMIP
institution_id         IPSL
source_id              IPSLCM6-ESMCO2
experiment_id          piControl
variant_label          r1i1p1f1
grid_label             g112
nominal_resolution     100 km
region                 glb
frequency              mon
realm                  ocean
variable_id            thetao
branded_variable       thetao_tavg-ol-hxy-sea
branding_suffix        tavg-ol-hxy-sea
temporal_label         tavg
vertical_label         ol
horizontal_label       hxy
area_label             sea
data_specs_version     MIP-DS7.1.0.0
table_id               <absent>   <-- gone in CMIP7


## Shape 1 — multi-file concat (`ficeberg`, 3hr)

`virtualize_from_urls` now accepts `pathlib.Path` and bare path strings directly; they
are normalized to `file://` URLs by `cmip7_virtualization.virtualize.as_url`, because
virtualizarr's `ObjectStoreRegistry` refuses any URL without a scheme.

In [8]:
print(as_url(GROUPS["ficeberg"][0]))

file:///Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data/ficeberg_tavg-ol-hxy-sea_3hr_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185001010130-185004302230.nc


In [9]:
vds_ficeberg, reg_ficeberg = virtualize_from_urls(GROUPS["ficeberg"])
vds_ficeberg

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


<xarray.Dataset> Size: 939MB
Dimensions:         (y: 332, x: 362, nvertex: 4, time: 1944, axis_nbounds: 2)
Coordinates:
  * time            (time) datetime64[ns] 16kB 1850-01-01T01:30:00 ... 1850-0...
    nav_lat         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
    nav_lon         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
Dimensions without coordinates: y, x, nvertex, axis_nbounds
Data variables:
    bounds_nav_lon  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    bounds_nav_lat  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    time_bounds     (time, axis_nbounds) float64 31kB ManifestArray<shape=(19...
    ficeberg        (time, y, x) float32 935MB ManifestArray<shape=(1944, 332...
Attributes: (12/43)
    name:                  /ccc/work/cont003/gencmip7/p86caub/IGCM_OUT/ESMCO2...
    description:           Created by xios
    title:                 Created by xios
    creation_date:         2026-05-30T20:23:10Z
    tracking_id:           hdl:21.14107/4629d039-391f-483d-94d6-d098f60d5efe
    Conventions:           CF-1.11 CF-1.12 CF-1.13
    ...                    ...
    physical_parameter:    ficeberg
    dr_version:            Software v1.4 - Content v1.2.2.3
    cv_version:            ESGVOC version 4.0.1 - CV content cmip7 1.1.0
    EXPID:                 piControl
    dr2xml_md5sum:         10a14c0903f85756f365df30a9a16531
    model_version:         6.1

In [10]:
print("registry keys:", list(reg_ficeberg.map.keys()))
n = sum(
    len(v.data.manifest.dict())
    for v in vds_ficeberg.variables.values()
    if hasattr(v.data, "manifest")
)
print("virtual chunk references:", n)

registry keys: [UrlKey(scheme='file', netloc='')]
virtual chunk references: 3892


### Gotcha 1 — static grid variables get broadcast along time

IPSL/NEMO output carries the curvilinear cell-corner arrays `bounds_nav_lon` and
`bounds_nav_lat` as `(y, x, nvertex)` **data variables**, not coordinates. xarray's
`combine_by_coords` default is `data_vars="all"`, which broadcasts every variable
lacking the concat dimension *along* it. Concatenating the two 3-hourly files turned
each 2 MB bounds array into a `(1944, y, x, nvertex)` 4 GB array whose manifest repeats
the same handful of chunk references 1944 times.

`virtualize_from_urls` therefore defaults to `data_vars="minimal"`. Both are shown here.

In [11]:
vds_all, _ = virtualize_from_urls(GROUPS["ficeberg"], data_vars="all")

for label, v in [
    ("minimal (our default)", vds_ficeberg),
    ("all (xarray default)", vds_all),
]:
    b = v["bounds_nav_lon"]
    print(
        f"{label:24s} dims={b.dims!s:36s} size={b.nbytes / 1e6:9.1f} MB "
        f"chunk_refs={len(b.data.manifest.dict())}"
    )

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


minimal (our default)    dims=('y', 'x', 'nvertex')                size=      1.9 MB chunk_refs=1
all (xarray default)     dims=('time', 'y', 'x', 'nvertex')        size=   3738.2 MB chunk_refs=1944


/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


## Shape 2 — single file with a vertical coordinate (`thetao`, mon, 1.46 GB)

### Gotcha 2 — every dimension coordinate must be loadable

`open_virtual_mfdataset` always routes through `combine_by_coords`, which needs a
pandas index for each dimension coordinate. A `ManifestArray` has no index, so any
dimension coordinate left virtual aborts the combine — **even for a single file**.
`thetao` has `olevel` (75 ocean levels) alongside `time`.

The cell below reproduces the failure on purpose by restricting `loadable_variables`
to `time`. The traceback is real and is kept in the notebook.

In [9]:
virtualize_from_urls(GROUPS["thetao"], loadable_variables=["time"])

ValueError: Every dimension requires a corresponding 1D coordinate and index for inferring concatenation order but the coordinate 'olevel' has no corresponding index

That coordinate is a dimension coordinate left as a virtual ManifestArray, so combine_by_coords has no index to order on. Add its name to `loadable_variables` (currently ['time']) or to cmip7_virtualization.virtualize.DEFAULT_LOADABLE_VARIABLES.

`olevel` is now in `DEFAULT_LOADABLE_VARIABLES`, so the default call works. Names absent
from a given file are ignored, so the list can safely be a superset.

TODO: I think we might want to revert this and instead implement https://github.com/carbonplan/cmip7-virtualization/issues/30 to avoid having to define all these default names and the inevitatbility that one file will have some other naming?

In [10]:
print(DEFAULT_LOADABLE_VARIABLES)
vds_thetao, reg_thetao = virtualize_from_urls(GROUPS["thetao"])
vds_thetao

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


('time', 'lat', 'lon', 'olevel', 'lev', 'depth', 'plev', 'height', 'sdepth')


<xarray.Dataset> Size: 4GB
Dimensions:         (olevel: 75, time: 120, y: 332, x: 362, nvertex: 4,
                     axis_nbounds: 2)
Coordinates:
  * olevel          (olevel) float32 300B 0.5058 1.556 ... 5.698e+03 5.902e+03
  * time            (time) datetime64[ns] 960B 1850-01-16T12:00:00 ... 1859-1...
    nav_lat         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
    nav_lon         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
Dimensions without coordinates: y, x, nvertex, axis_nbounds
Data variables:
    bounds_nav_lon  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    bounds_nav_lat  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    olevel_bounds   (olevel, axis_nbounds) float32 600B ManifestArray<shape=(...
    time_bounds     (time, axis_nbounds) float64 2kB ManifestArray<shape=(120...
    thetao          (time, olevel, y, x) float32 4GB ManifestArray<shape=(120...
Attributes: (12/43)
    name:                  /ccc/work/cont003/gencmip7/p86caub/IGCM_OUT/ESMCO2...
    description:           Created by xios
    title:                 Created by xios
    creation_date:         2026-05-30T20:37:47Z
    tracking_id:           hdl:21.14107/828c7ada-42a8-40c7-9389-3390e9a3044f
    Conventions:           CF-1.11 CF-1.12 CF-1.13
    ...                    ...
    physical_parameter:    thetao
    dr_version:            Software v1.4 - Content v1.2.2.3
    cv_version:            ESGVOC version 4.0.1 - CV content cmip7 1.1.0
    EXPID:                 piControl
    dr2xml_md5sum:         10a14c0903f85756f365df30a9a16531
    model_version:         6.1

## Shape 3 — `fx`, no time dimension (`areacello`)

The awkward case: nothing to concatenate, and none of the default loadable variables
exist in the file (the grid is curvilinear, so the coordinates are 2-D `nav_lat` /
`nav_lon`, not 1-D `lat` / `lon`). It works anyway — there are no dimension coordinates,
so `combine_by_coords` has nothing to index.

In [11]:
vds_areacello, reg_areacello = virtualize_from_urls(GROUPS["areacello"])
vds_areacello

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


<xarray.Dataset> Size: 5MB
Dimensions:         (y: 332, x: 362, nvertex: 4)
Coordinates:
    nav_lat         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
    nav_lon         (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
Dimensions without coordinates: y, x, nvertex
Data variables:
    bounds_nav_lon  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    bounds_nav_lat  (y, x, nvertex) float32 2MB ManifestArray<shape=(332, 362...
    areacello       (y, x) float32 481kB ManifestArray<shape=(332, 362), dtyp...
Attributes: (12/42)
    name:                  /ccc/work/cont003/gencmip7/p86caub/IGCM_OUT/ESMCO2...
    description:           Created by xios
    title:                 Created by xios
    creation_date:         2026-05-30T20:30:55Z
    tracking_id:           hdl:21.14107/9980076b-7b5d-4c3f-a867-35d7fab97bb9
    Conventions:           CF-1.11 CF-1.12 CF-1.13
    ...                    ...
    physical_parameter:    areacello
    dr_version:            Software v1.4 - Content v1.2.2.3
    cv_version:            ESGVOC version 4.0.1 - CV content cmip7 1.1.0
    EXPID:                 piControl
    dr2xml_md5sum:         10a14c0903f85756f365df30a9a16531
    model_version:         6.1

## Writing the Icechunk stores

For a local source the virtual-chunk container needs a `file:///dir/` prefix.
`storage.local_url_prefix(paths)` derives it from the source paths, because the
virtualizarr registry only keys on `(scheme, netloc)` — which for a local store is the
bare `file://` — and Icechunk **rejects** a filesystem-root prefix outright
(`ValueError: Url prefix for file:// containers must include a path`).

In [12]:
print(local_url_prefix(GROUPS["ficeberg"]))

file:///Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data/


In [13]:
def write_store(name, vds, registry, paths):
    """Write *vds* to a local Icechunk repo under STORE_ROOT and return its path."""
    prefixes = [local_url_prefix(paths)]
    config = ic.RepositoryConfig.default()
    for vcc in vccs_from_registry(registry, local_prefixes=prefixes):
        config.set_virtual_chunk_container(vcc)
    auth = authorize_prefixes_from_registry(registry, local_prefixes=prefixes)

    repo_path = STORE_ROOT / name
    repo = ic.Repository.open_or_create(
        storage=ic.local_filesystem_storage(str(repo_path)),
        config=config,
        authorize_virtual_chunk_access=auth,
    )
    session = repo.writable_session("main")
    vds.vz.to_icechunk(session.store)
    snapshot = session.commit(f"virtual references for {name}")
    repo.save_config()
    return repo_path, snapshot, auth


written = {}
for name, vds, registry in [
    ("ficeberg", vds_ficeberg, reg_ficeberg),
    ("thetao", vds_thetao, reg_thetao),
    ("areacello", vds_areacello, reg_areacello),
]:
    repo_path, snapshot, auth = write_store(name, vds, registry, GROUPS[name])
    written[name] = (repo_path, auth)
    print(f"{name:10s} {snapshot}  {repo_path}")

  2026-07-26T19:59:33.693006Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



  2026-07-26T19:59:33.888230Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



ficeberg   SKVWTDS9Q5QEZSRW775G  /var/folders/3b/rlk8rxgs1xx44kmtzfrwlvth0000gn/T/cmip7-ipsl-icechunk/ficeberg
thetao     7Y588Z37JGSJFMV76610  /var/folders/3b/rlk8rxgs1xx44kmtzfrwlvth0000gn/T/cmip7-ipsl-icechunk/thetao
areacello  QDKER1B4YGH7QTYHF010  /var/folders/3b/rlk8rxgs1xx44kmtzfrwlvth0000gn/T/cmip7-ipsl-icechunk/areacello


  2026-07-26T19:59:33.911369Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



### Store sizes

Virtual references only, so the stores should be a rounding error next to the 1.6 GB of
source NetCDF.

In [14]:
src_total = sum(p.stat().st_size for paths in GROUPS.values() for p in paths)
store_total = 0
for name, (repo_path, _) in written.items():
    size = sum(f.stat().st_size for f in repo_path.rglob("*") if f.is_file())
    store_total += size
    print(f"{name:10s} {size / 1e6:8.2f} MB")
print(f"\ntotal store  {store_total / 1e6:8.2f} MB")
print(f"total source {src_total / 1e6:8.2f} MB")
print(f"ratio        {store_total / src_total:.4%}")

ficeberg       0.10 MB
thetao         0.02 MB
areacello      0.01 MB

total store      0.12 MB
total source  1630.58 MB
ratio        0.0074%


## Round-trip verification

Reopen each store and compare **actual array values** against a direct
`xr.open_dataset` of the source NetCDF. `thetao` and `ficeberg` are too large to
materialize whole (4 GB and 0.9 GB), so those comparisons take slices — chosen to cover
the interesting parts: the file boundary for the concatenated case, and the top and
bottom of the water column for the 4-D case. `areacello` is compared in full.

In [15]:
def reopen(name):
    repo_path, auth = written[name]
    repo = ic.Repository.open(
        storage=ic.local_filesystem_storage(str(repo_path)),
        authorize_virtual_chunk_access=auth,
    )
    return xr.open_zarr(
        repo.readonly_session("main").store, consolidated=False, zarr_format=3
    )

In [16]:
# --- fx: full array comparison -------------------------------------------------
ic_area = reopen("areacello")
nc_area = xr.open_dataset(GROUPS["areacello"][0], engine="h5netcdf")

np.testing.assert_array_equal(ic_area["areacello"].values, nc_area["areacello"].values)
np.testing.assert_array_equal(ic_area["nav_lat"].values, nc_area["nav_lat"].values)
np.testing.assert_array_equal(
    ic_area["bounds_nav_lon"].values, nc_area["bounds_nav_lon"].values
)
print("areacello: full-array match on areacello, nav_lat, bounds_nav_lon")
print("shape", ic_area["areacello"].shape, "sum", float(ic_area["areacello"].sum()))
nc_area.close()

areacello: full-array match on areacello, nav_lat, bounds_nav_lon
shape (332, 362) sum 363577168887808.0


  2026-07-26T19:59:33.927533Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


In [17]:
# --- multi-file concat: the file boundary is the thing to check ---------------
ic_fice = reopen("ficeberg")
nc_fice = xr.open_mfdataset(
    GROUPS["ficeberg"],
    engine="h5netcdf",
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

# time axis in full: proves the two files were ordered and joined correctly
np.testing.assert_array_equal(ic_fice["time"].values, nc_fice["time"].values)
n0 = xr.open_dataset(GROUPS["ficeberg"][0], engine="h5netcdf").sizes["time"]
print(f"time: {ic_fice.sizes['time']} steps, boundary after {n0} (file 0)")
print("first", ic_fice["time"].values[0], " last", ic_fice["time"].values[-1])

# values spanning the boundary, plus the very first and very last timestep
sl = slice(n0 - 5, n0 + 5)
np.testing.assert_array_equal(
    ic_fice["ficeberg"].isel(time=sl).values, nc_fice["ficeberg"].isel(time=sl).values
)
for i in (0, -1):
    np.testing.assert_array_equal(
        ic_fice["ficeberg"].isel(time=i).values,
        nc_fice["ficeberg"].isel(time=i).values,
    )
print("ficeberg: values match across the file boundary and at both ends")

  2026-07-26T19:59:34.068113Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


time: 1944 steps, boundary after 960 (file 0)
first 1850-01-01T01:30:00.000000000  last 1850-08-31T22:30:00.000000000


ficeberg: values match across the file boundary and at both ends


In [18]:
# Iceberg meltwater flux is zero over most of the ocean, so "values are equal" is a weak
# claim unless we look somewhere it is not. Pull a genuinely non-zero point out of the
# SECOND file, through the store, and compare against that file opened directly.
nc1 = xr.open_dataset(GROUPS["ficeberg"][1], engine="h5netcdf")
t_local = 10
field = nc1["ficeberg"].isel(time=t_local).values
nz = np.argwhere(np.nan_to_num(field) != 0)
print(f"non-zero cells at t={t_local} in file 1: {len(nz)} of {field.size}")

y, x = (int(v) for v in nz[len(nz) // 2])
store_val = float(ic_fice["ficeberg"].isel(time=n0 + t_local, y=y, x=x).values)
src_val = float(field[y, x])
print(
    f"y={y} x={x}  store={store_val!r}  source file 1={src_val!r}  equal={store_val == src_val}"
)
assert store_val == src_val

# and the whole non-zero population of that timestep
a = ic_fice["ficeberg"].isel(time=n0 + t_local).values
np.testing.assert_array_equal(a, field)
print(f"full timestep matches; sum={np.nansum(field):.6e}, max={np.nanmax(field):.6e}")
nc1.close()
nc_fice.close()

non-zero cells at t=10 in file 1: 15767 of 120184
y=87 x=300  store=1.4276275474289779e-11  source file 1=1.4276275474289779e-11  equal=True
full timestep matches; sum=1.645377e-02, max=1.286537e-04


In [19]:
# --- 4-D single file: top and bottom of the water column ----------------------
ic_th = reopen("thetao")
nc_th = xr.open_dataset(GROUPS["thetao"][0], engine="h5netcdf")

np.testing.assert_array_equal(ic_th["time"].values, nc_th["time"].values)
np.testing.assert_array_equal(ic_th["olevel"].values, nc_th["olevel"].values)
print("thetao dims:", dict(ic_th.sizes))

for sel in [
    dict(time=slice(0, 3)),  # first three months, all 75 levels
    dict(olevel=0),  # surface, all months
    dict(olevel=74),  # deepest level, all months
    dict(time=-1, olevel=slice(60, 75)),
]:
    a = ic_th["thetao"].isel(**sel).values
    b = nc_th["thetao"].isel(**sel).values
    np.testing.assert_array_equal(a, b)
    print(f"  match {sel!s:42s} {a.shape} nbytes={a.nbytes / 1e6:.1f} MB")
nc_th.close()
print("thetao: all sampled slices match the source NetCDF")

thetao dims: {'y': 332, 'x': 362, 'nvertex': 4, 'time': 120, 'olevel': 75, 'axis_nbounds': 2}


  2026-07-26T19:59:34.299139Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

/Users/juliusbusecke/Code/cmip7-virtualization/.claude/worktrees/agent-a230ea7a9172b49af/.venv/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


  match {'time': slice(0, 3, None)}                (3, 75, 332, 362) nbytes=108.2 MB


  match {'olevel': 0}                              (120, 332, 362) nbytes=57.7 MB


  match {'olevel': 74}                             (120, 332, 362) nbytes=57.7 MB
  match {'time': -1, 'olevel': slice(60, 75, None)} (15, 332, 362) nbytes=7.2 MB
thetao: all sampled slices match the source NetCDF


---

# Findings

**Local Icechunk generation works for all three shapes.** Multi-file concat, single
4-D file, and `fx` with no time dimension all wrote and read back with matching values.
Three library changes were needed.

1. **Local paths were not supported.** `ObjectStoreRegistry` raises on a URL with no
   scheme, so a bare path or `pathlib.Path` could not reach `open_virtual_mfdataset`.
   `virtualize.as_url` now normalizes paths to absolute `file://` URLs (resolving `~`
   and relative paths, since the URL is baked verbatim into the chunk manifest), and
   `virtualize_from_urls` registers an obstore `LocalStore` rooted at `/`. It also skips
   the read-through cache and the eager whole-file reader for local sources — reading a
   1.46 GB NetCDF fully into RAM to parse its headers is pure waste when a seek is free.

2. **Icechunk rejects `file:///` as a virtual-chunk-container prefix.** It insists on a
   real directory ending in `/`. The registry key for a local store carries only
   `(scheme, netloc)`, so the directory has to be recovered from the source paths —
   hence `storage.local_url_prefix()`, and the new `local_prefixes=` argument on
   `vccs_from_registry` / `authorize_prefixes_from_registry`. The error surfaces on
   `set_virtual_chunk_container`, not on the `VirtualChunkContainer` constructor.

3. **`data_vars="all"` inflated the store.** IPSL/NEMO carries `bounds_nav_lon` and
   `bounds_nav_lat` as static `(y, x, nvertex)` *data variables*. xarray's default
   broadcast them along time, turning 2 MB into 4 GB of duplicate manifest entries.
   `virtualize_from_urls` now defaults to `data_vars="minimal"`.

And one that was a data property rather than a bug:

4. **Every dimension coordinate must be in `loadable_variables`.** `combine_by_coords`
   needs a pandas index, and a `ManifestArray` has none — so `olevel` had to be added.
   This bites single-file datasets too, because the combine still runs. The old hard-coded
   `["time", "lon", "lat", "something_that_i_never_expect"]` is now
   `DEFAULT_LOADABLE_VARIABLES`, overridable per call, and the cryptic xarray error is
   re-raised with the fix spelled out.

## CMIP7 branded-variable filenames

Nothing broke, because nothing in the package parses filenames. What they *imply*:

- The field layout changed and gained two fields, so any CMIP6-shaped splitter
  mis-parses them — position 1 is now the branding suffix, not `table_id`.
- **`table_id` is gone from the global attributes entirely.** CMIP6-style instance ids
  have no value to put in that slot, and no slot for the branding suffix. STAC item-id
  construction needs a decision here.
- `branded_variable` (`thetao_tavg-ol-hxy-sea`) is the natural per-dataset key, and it
  is available as a global attribute, so filename parsing can be avoided.
- The curvilinear ocean grid means coordinates are 2-D `nav_lat`/`nav_lon`; there are no
  1-D `lat`/`lon` dimension coordinates at all.

## Caveats

- `thetao` and `ficeberg` were verified on slices, not in full — 4 GB and 0.9 GB
  respectively. The slices cover the file boundary and both ends of the time and depth
  axes, which is where concatenation errors show up.
- A local store is not publishable: its manifests hold absolute paths from one machine.
  Rebuilding against OSN-hosted sources is the fix; that scaffolding lives in
  `notebooks/testing/osn-ref-generation-ipsl.ipynb` and is unexecuted.
- Writing is single-threaded here; Icechunk warns that local-filesystem storage is not
  safe for concurrent commits.